In [ ]:
!pip install sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.4 MB/s eta 0:00:00


In [ ]:
documents = [
    "Employees are entitled to 12 casual leave days per year.",

    "Employees are entitled to 18 days of annual leave per year.",

    "Employees may work from home with manager approval.",

    "The work from home policy allows eligible employees to work remotely.",

    "Employees must submit travel reimbursement claims within 30 days.",

    "Travel expenses must be supported by valid bills and receipts.",

    "The employee PF contribution is 12 percent of the basic salary.",

    "Employees can apply for sick leave when they are medically unwell.",

    "Maternity leave is available to eligible employees according to company policy.",

    "Employees must submit leave applications through the HR portal.",

    "Leave applications require manager approval.",

    "Leave encashment is permitted according to company policy."
]

In [ ]:
for i, doc in enumerate(documents):
    print(i, ":", doc)

0 : Employees are entitled to 12 casual leave days per year.
1 : Employees are entitled to 18 days of annual leave per year.
2 : Employees may work from home with manager approval.
3 : The work from home policy allows eligible employees to work remotely.
4 : Employees must submit travel reimbursement claims within 30 days.
5 : Travel expenses must be supported by valid bills and receipts.
6 : The employee PF contribution is 12 percent of the basic salary.
7 : Employees can apply for sick leave when they are medically unwell.
8 : Maternity leave is available to eligible employees according to company policy.
9 : Employees must submit leave applications through the HR portal.
10 : Leave applications require manager approval.
11 : Leave encashment is permitted according to company policy.


Keyword search asks:

"Which documents contain words similar to the words in my question?"

For example:

Query:
casual leave entitlement

Important words:

casual
leave
entitlement

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
tokenized_documents = [
    doc.lower().split()
    for doc in documents
]

In [ ]:
bm25 = BM25Okapi(tokenized_documents)

In [ ]:
query = "casual leave entitlement"

query_tokens = query.lower().split()

scores = bm25.get_scores(query_tokens)

In [ ]:
for i, score in enumerate(scores):
    print(i, score, documents[i])

0 2.409145177458611 Employees are entitled to 12 casual leave days per year.
1 0.40084113432974094 Employees are entitled to 18 days of annual leave per year.
2 0.0 Employees may work from home with manager approval.
3 0.0 The work from home policy allows eligible employees to work remotely.
4 0.0 Employees must submit travel reimbursement claims within 30 days.
5 0.0 Travel expenses must be supported by valid bills and receipts.
6 0.0 The employee PF contribution is 12 percent of the basic salary.
7 0.40084113432974094 Employees can apply for sick leave when they are medically unwell.
8 0.40084113432974094 Maternity leave is available to eligible employees according to company policy.
9 0.4397367700059422 Employees must submit leave applications through the HR portal.
10 0.545626560776604 Leave applications require manager approval.
11 0.46215960813655693 Leave encashment is permitted according to company policy.


In [ ]:
ranked_indices = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)

In [ ]:
for i in ranked_indices[:3]:
    print(
        f"Score: {scores[i]:.2f} | {documents[i]}"
    )

Score: 2.41 | Employees are entitled to 12 casual leave days per year.
Score: 0.55 | Leave applications require manager approval.
Score: 0.46 | Leave encashment is permitted according to company policy.


BM25 is useful when exact terms matter.

Examples:

Policy code
Employee ID
PF
HR-PF-2026
Specific terminology
Names

In [ ]:
#Semantic Vector Search

In [ ]:
#Semantic search tries to understand meaning similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = model.encode(
    documents,
    normalize_embeddings=True
)

In [ ]:
print(embeddings.shape)

(12, 384)


In [ ]:
import faiss
import numpy as np

embedding_matrix = np.array(
    embeddings
).astype("float32")

index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

index.add(embedding_matrix)

In [ ]:
query = "How many casual leaves can an employee take?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)

query_embedding = np.array(
    query_embedding
).astype("float32")

In [ ]:
#search

In [ ]:
scores, indices = index.search(
    query_embedding,
    3
)

In [ ]:
for score, idx in zip(scores[0], indices[0]):

    print(
        f"Score: {score:.3f} | {documents[idx]}"
    )

Score: 0.692 | Employees are entitled to 12 casual leave days per year.
Score: 0.520 | Employees are entitled to 18 days of annual leave per year.
Score: 0.393 | Employees must submit leave applications through the HR portal.


In [ ]:
#Top-K Search

In [ ]:
#k most relevant chunks

In [ ]:
k = 3

In [ ]:
query = "work from home policy"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    query_embedding,
    k=3
)

In [ ]:
for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    print(
        f"{rank}. Score={score:.3f}"
    )

    print(documents[idx])
    print()

1. Score=0.788
The work from home policy allows eligible employees to work remotely.

2. Score=0.683
Employees may work from home with manager approval.

3. Score=0.335
Leave encashment is permitted according to company policy.



In [ ]:
#threshhold

Threshold Search

Now change the question:

Instead of saying:

Give me 3 results.

Say:

Give me only results whose similarity is greater than 0.75.

In [ ]:
query = "travel reimbursement"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    query_embedding,
    len(documents)
)

In [ ]:
threshold = 0.75

for score, idx in zip(scores[0], indices[0]):

    if score >= threshold:

        print(
            f"Score: {score:.3f}"
        )

        print(documents[idx])
        print()

Hybrid Search

Now we combine:

Keyword Search
       +
Semantic Search

In [ ]:
query = "PF contribution percentage"

query_tokens = query.lower().split()

keyword_scores = bm25.get_scores(
    query_tokens
)

In [ ]:
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

vector_scores, vector_indices = index.search(
    query_embedding,
    len(documents)
)

In [ ]:
vector_score_array = np.zeros(
    len(documents)
)

for score, idx in zip(
    vector_scores[0],
    vector_indices[0]
):

    vector_score_array[idx] = score

In [ ]:
keyword_score_array = np.array(
    keyword_scores
)

keyword_normalized = (
    keyword_score_array -
    keyword_score_array.min()
) / (
    keyword_score_array.max() -
    keyword_score_array.min() + 1e-10
)

In [ ]:
#combine

In [ ]:
alpha = 0.5

In [ ]:
hybrid_scores = (
    alpha * keyword_normalized
    +
    (1 - alpha) * vector_score_array
)